# 04.01 章节概述：VLA 与 LeRobot 机械臂训练

<img src="./images/vla_pipeline.png" width="700">

## 本章节面向谁

本章带你进入**具身智能（Embodied AI）**领域，使用开源框架 **LeRobot** 训练 SO-101 机械臂的 **ACT 策略**，完成"视觉观察 → 动作生成"的方块抓取任务。这是前 3 章（视觉/语音/语言）的延伸：把 AI 模型装进物理机器人，让它学会操作真实世界。

## 什么是 VLA

**VLA（Vision-Language-Action）** 是一类让机器人通过**观察（视觉）**和**指令（语言）**学会**操作（动作）**的模型。本章节聚焦其中最基础的形式——**纯视觉模仿学习**：机器人观察摄像头画面，模仿人类演示的动作。

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">要素</th><th align="left">本章节实现</th></tr>
<tr><td align="left">视觉（Vision）</td><td align="left">前置摄像头 + 腕部摄像头的 RGB 图像</td></tr>
<tr><td align="left">动作（Action）</td><td align="left">6 自由度机械臂的关节角度 + 夹爪开合</td></tr>
<tr><td align="left">策略（Policy）</td><td align="left">ACT（Action Chunking with Transformers）</td></tr>
</table>

## LeRobot 与 SO-101 简介

- **LeRobot**：HuggingFace 开源的机器人学习框架，提供数据采集、训练、评估的全流程工具，内置 ACT、Diffusion、π0 等策略。
- **SO-101**：Seeed Studio 设计的 6 自由度开源机械臂，配合 LeRobot 可低成本完成模仿学习实验。

## 学习前置要求

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">类别</th><th align="left">具体要求</th></tr>
<tr><td align="left">编程基础</td><td align="left">熟悉 Python，了解命令行操作</td></tr>
<tr><td align="left">深度学习</td><td align="left">了解 CNN（ResNet）、Transformer、训练/推理的概念</td></tr>
<tr><td align="left">前序章节</td><td align="left">建议先学第 1 章（视觉基础）和第 3 章（模型训练）</td></tr>
</table>

## 章节目标

学完本章后，你将能够：

1. 理解 **VLA** 和 **模仿学习** 的核心思想，说清 ACT 策略的架构；
2. 理解**遥操数据采集**的流程（理论层面）；
3. 加载 LeRobot 数据集并探索其结构（episode、帧、动作）；
4. 用 `lerobot-train` 训练 ACT 策略，解读训练过程曲线；
5. 对训练后的模型做离线测试，理解真机部署流程，包括部署到**香橙派（昇腾 310 NPU）**的方法。

## 章节内容与跳转链接

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">小节</th><th align="left">主题</th><th align="left">链接</th></tr>
<tr><td align="left">04.01</td><td align="left">章节概述（本节）</td><td align="left">-</td></tr>
<tr><td align="left">04.02</td><td align="left">VLA 原理、数据采集理论与数据集探索</td><td align="left"><a href="./04.02_theory_data.ipynb">前往</a></td></tr>
<tr><td align="left">04.03</td><td align="left">ACT 训练与过程可视化</td><td align="left"><a href="./04.03_training.ipynb">前往</a></td></tr>
<tr><td align="left">04.04</td><td align="left">离线测试、真机推理理论与章节实践</td><td align="left"><a href="./04.04_eval_practice.ipynb">前往</a></td></tr>
</table>

## 环境要求

本章**必须在昇腾 NPU 环境下训练**（ACT 模型有 5200 万参数，CPU 无法胜任）。

<table style="text-align: left; margin-left: 0;">
<tr><th align="left">环境</th><th align="left">说明</th></tr>
<tr><td align="left">☁️ **CANNLab 云上（唯一推荐）**</td><td align="left">选择 **Python 3.11.4 (CANN)** 内核，含昇腾 NPU + torch_npu，训练与推理均可</td></tr>
</table>

**计算设备**：
- 昇腾 NPU（CANNLab 首选且唯一推荐）
- CPU（仅用于数据加载验证，训练会很慢，不推荐）
- notebook 会自动检测设备，NPU 训练通过课程集成的补丁脚本完成（见 04.03）

> ⚠️ LeRobot 官方不支持昇腾 NPU，本章提供了基于 CANN 官方样例裁剪的 NPU 补丁脚本（已验证通过），详见 04.03 和 `src/npu_support/SETUP_NPU.md`。

## 数据集说明

本章使用 SO-101 机械臂的**方块抓取数据集**：
- **规模**：100 episodes / 68146 帧
- **内容**：前置摄像头 + 腕部摄像头的视频，配合 6 维动作标注
- **格式**：LeRobot v3（parquet + mp4 视频）
- **大小**：约 449MB
- **获取**：运行 04.02 的 notebook 时会**自动下载**到 `src/data_final/` 目录（首次下载约 3-10 分钟），无需手动准备。